In [6]:
import warnings
import pandas as pd
import numpy as np
import os
from datetime import datetime

# Suppress all FutureWarnings
warnings.simplefilter(action='ignore', category=FutureWarning)

def format_duration(duration):
    total_seconds = int(duration.total_seconds())
    hours, remainder = divmod(total_seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{hours:02}:{minutes:02}:{seconds:02}"

def check_tp_sl(price_data, entry_datetime, tp_price, sl_price, side):
    subsequent_prices = price_data.loc[entry_datetime:]
    for current_datetime, price_row in subsequent_prices.iterrows():
        if side == 'Buy':
            if price_row['High'] >= tp_price:
                return 1, current_datetime - entry_datetime
            elif price_row['Low'] <= sl_price:
                return -1, current_datetime - entry_datetime
        else:
            if price_row['Low'] <= tp_price:
                return 1, current_datetime - entry_datetime
            elif price_row['High'] >= sl_price:
                return -1, current_datetime - entry_datetime
    return 0, None

def determine_entry(price_data, signal_datetime, percentage_change, side, entry_time_offset, time_limit_minutes):
    signal_open_price = price_data.at[signal_datetime, 'Open']
    percentage_change_price = signal_open_price * (1 - percentage_change) if side == 'Buy' else signal_open_price * (1 + percentage_change)
    entry_datetime_offset = signal_datetime + pd.Timedelta(minutes=entry_time_offset)
    time_limit = signal_datetime + pd.Timedelta(minutes=time_limit_minutes)
    subsequent_prices = price_data.loc[signal_datetime:time_limit]

    for current_datetime, price_row in subsequent_prices.iterrows():
        if side == 'Buy' and price_row['Low'] <= percentage_change_price:
            return current_datetime, percentage_change_price, current_datetime - signal_datetime
        elif side == 'Sell' and price_row['High'] >= percentage_change_price:
            return current_datetime, percentage_change_price, current_datetime - signal_datetime

    if entry_datetime_offset in price_data.index:
        entry_price = price_data.at[entry_datetime_offset, 'Open']
        return entry_datetime_offset, entry_price, entry_datetime_offset - signal_datetime

    return None, None, None

def backtest_trades(price_data, signal_data, tp, sl, entry_time_offset, percentage_change, entry_methods, time_limit_minutes):
    output_data = []
    current_margin = initial_margin = 100000
    exit_datetimes = []

    for i, row in signal_data.iterrows():
        signal_datetime = row['Datetime']
        signal_value = row['Signal']
        side = 'Buy' if signal_value > 0 else 'Sell' if signal_value < 0 else None

        if side is None:
            continue

        signal_open_price = price_data.at[signal_datetime, 'Open']
        exit_datetimes.sort()

        later_exits = [ed for ed in exit_datetimes if ed[0] > signal_datetime]
        if len(later_exits) >= 3 or (len(later_exits) == 2 and later_exits[-1][1] != side) or (len(later_exits) == 1 and later_exits[-1][1] == side):
            output_data.append({
                'Datetime': signal_datetime,
                'Side': side,
                'Signal Open Price': signal_open_price,
                'Entry Price': None,
                'TP Price': None,
                'SL Price': None,
                'Result': 'Ignored',
                'Duration': '00:00:00',
                'Execution Latency': '00:00:00',
                'ROI': 0,
                'NAV': current_margin,
                'Ignore Reason': 'Ignored due to exit datetime constraints'
            })
            continue

        entry_datetime, entry_price, entry_duration = determine_entry(price_data, signal_datetime, percentage_change, side, entry_time_offset, time_limit_minutes)

        if entry_datetime is None:
            output_data.append({
                'Datetime': signal_datetime,
                'Side': side,
                'Signal Open Price': signal_open_price,
                'Entry Price': None,
                'TP Price': None,
                'SL Price': None,
                'Result': 'Not Filled',
                'Duration': '00:00:00',
                'Execution Latency': '00:00:00',
                'ROI': 0,
                'NAV': current_margin,
                'Ignore Reason': 'Entry conditions not met'
            })
            continue

        tp_price = entry_price * (1 + tp) if side == 'Buy' else entry_price * (1 - tp)
        sl_price = entry_price * (1 - sl) if side == 'Buy' else entry_price * (1 + sl)

        result, duration = check_tp_sl(price_data, entry_datetime, tp_price, sl_price, side)

        if result in [1, -1]:
            exit_datetimes.append((entry_datetime + duration, side))

        if result == 1:
            current_margin *= (1 + tp)
        elif result == -1:
            current_margin *= (1 - sl)

        roi = ((current_margin - initial_margin) / initial_margin) * 100
        nav = current_margin
        initial_margin = current_margin

        output_data.append({
            'Datetime': signal_datetime,
            'Side': side,
            'Signal Open Price': signal_open_price,
            'Entry Price': entry_price,
            'TP Price': tp_price,
            'SL Price': sl_price,
            'Result': result,
            'Duration': format_duration(duration),
            'Execution Latency': format_duration(entry_duration),
            'ROI': roi,
            'NAV': nav,
            'Ignore Reason': ''
        })

    return pd.DataFrame(output_data)

def calculate_metrics(group, initial_nav):
    total_trades = len(group)
    total_wins = len(group[group['Result'] == 1])
    total_losses = len(group[group['Result'] == -1])
    win_rate = total_wins / total_trades if total_trades > 0 else 0

    final_nav = group['NAV'].iloc[-1] if total_trades > 0 else initial_nav
    roi = ((final_nav - initial_nav) / initial_nav) * 100

    cumulative_returns = (group['NAV'] - initial_nav).cumsum()
    peak = cumulative_returns.cummax()
    drawdown = (peak - cumulative_returns).max()

    return {
        'Total Trades': total_trades,
        'Total Wins': total_wins,
        'Total Losses': total_losses,
        'Win Rate': win_rate,
        'ROI': roi,
        'NAV': final_nav,
        'Max Drawdown': drawdown
    }

def generate_report(price_data, signal_data, scenarios, output_directory):
    report_columns = ['Scenario', 'Period', 'Total Trades', 'Total Wins', 'Total Losses', 'Win Rate', 'ROI', 'NAV', 'Max Drawdown']
    report_data = pd.DataFrame(columns=report_columns)

    for scenario in scenarios:
        tp = scenario['tp']
        sl = scenario['sl']
        entry_time_offset = scenario['entry_time_offset']
        percentage_change = scenario['percentage_change']
        time_limit_minutes = scenario['time_limit_minutes']
        
        trade_data = backtest_trades(price_data, signal_data, tp, sl, entry_time_offset, percentage_change, scenario['entry_methods'], time_limit_minutes)
        
        monthly_groups = trade_data.groupby(trade_data['Datetime'].dt.to_period('M'))
        monthly_reports = []
        initial_nav = 100000
        for month, group in monthly_groups:
            metrics = calculate_metrics(group, initial_nav)
            metrics['Period'] = month.strftime('%Y-%m')
            metrics['Scenario'] = f"TP={tp}, SL={sl}, Offset={entry_time_offset}, pctChange={percentage_change}, TimeLimit={time_limit_minutes}"
            monthly_reports.append(pd.DataFrame([metrics]))
            initial_nav = metrics['NAV']

        if monthly_reports:
            monthly_report = pd.concat(monthly_reports, ignore_index=True)
            report_data = pd.concat([report_data, monthly_report], ignore_index=True)

        overall_metrics = calculate_metrics(trade_data, 100000)
        overall_metrics['Period'] = 'Overall'
        overall_metrics['Scenario'] = f"TP={tp}, SL={sl}, Offset={entry_time_offset}, pctChange={percentage_change}, TimeLimit={time_limit_minutes}"
        overall_report = pd.DataFrame([overall_metrics])
        report_data = pd.concat([report_data, overall_report], ignore_index=True)

    os.makedirs(output_directory, exist_ok=True)
    report_data.to_csv(os.path.join(output_directory, 'one_backtesting.csv'), index=False)

    return report_data


In [7]:
price_data = pd.read_csv('E:\Signal Backtesting\Input\price 2023-04-18, 2024-04-18.csv', parse_dates=['Datetime'],
                         index_col='Datetime')
# Re-load the signal data without setting the index
signal_data = pd.read_csv('E:\Signal Backtesting\Input\\2023_signal.csv', parse_dates=['Datetime'])

In [10]:
# Define the parameters
tp = 0.0096
sl = 0.014
entry_time_offset = 0  # Time offset in minutes
percentage_change = 0.00008
time_limit_minutes = 120
entry_methods = ['percentage_change', 'time_offset']

# Call the backtest_trades function
backtest_output = backtest_trades(
    price_data=price_data,
    signal_data=signal_data,
    tp=tp,
    sl=sl,
    entry_time_offset=entry_time_offset,
    percentage_change=percentage_change,
    entry_methods=entry_methods,
    time_limit_minutes=time_limit_minutes
)

backtest_output

,Datetime,Side,Signal Open Price,Entry Price,TP Price,SL Price,Result,Duration,Execution Latency,ROI,NAV,Ignore Reason
0,2023-04-18 14:58:00,Sell,30250.0,30252.420000,29961.996768,30675.953880,1,03:02:00,00:02:00,0.96,100960.000000,
1,2023-04-18 18:58:00,Sell,30130.5,30132.910440,29843.634500,30554.771186,1,13:10:00,00:01:00,0.96,101929.216000,
2,2023-04-18 22:58:00,Sell,30344.9,NaN,NaN,NaN,Ignored,00:00:00,00:00:00,0.00,101929.216000,Ignored due to exit datetime constraints
3,2023-04-19 10:58:00,Buy,29249.7,29247.360024,29528.134680,28837.896984,-1,10:45:00,00:00:00,-1.40,100502.206976,
4,2023-04-19 14:58:00,Buy,29380.6,NaN,NaN,NaN,Ignored,00:00:00,00:00:00,0.00,100502.206976,Ignored due to exit datetime constraints
...,...,...,...,...,...,...,...,...,...,...,...,...
349,2023-12-27 01:58:00,Buy,42480.3,42476.901576,42884.679831,41882.224954,1,08:26:00,00:00:00,0.96,90775.371443,
350,2023-12-28 04:58:00,Sell,43377.4,43380.870192,42964.413838,43988.202375,1,02:12:00,00:05:00,0.96,91646.815008,
351,2023-12-28 10:58:00,Sell,43148.1,43151.551848,42737.296950,43755.673574,1,01:58:00,00:04:00,0.96,92526.624433,
352,2023-12-29 10:58:00,Buy,42759.0,42755.579280,43166.032841,42157.001170,1,03:44:00,00:00:00,0.96,93414.880027,
